[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/groq_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪使用 Groq 构建的应用</h1>

Groq API 在调用托管 LLM 时提供更快的推理和更高的效率。Phoenix 将查询引擎每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 `span`，让 LLM 应用具备*可观测性*。

本教程将：
- 使用 Groq 发送 API 请求来回答问题
- 使用全局 `arize_phoenix` 处理器，以 [OpenInference 追踪](https://github.com/Arize-ai/openinference)格式记录数据
- 检查应用的追踪和 span，定位延迟和成本来源

ℹ️ 本 Notebook 需要 Groq API 密钥。

# 1. 安装依赖并配置环境
安装 Phoenix 和 Groq，将 API 密钥设置为环境变量。

In [ ]:
! pip install -q openinference-instrumentation-groq groq arize-phoenix

In [ ]:
import os
from getpass import getpass

if not (groq_api_key := os.getenv("GROQ_API_KEY")):
    groq_api_key = getpass("🔑 请输入 Groq API 密钥： ")

os.environ["GROQ_API_KEY"] = groq_api_key

## 2. 启动 Phoenix 并配置自动埋点

使用 `register` 将本地 Phoenix 注册为 OTel 端点。启动本地 Phoenix，收集会话中每次请求的追踪。

In [ ]:
from phoenix.otel import register

import phoenix as px

session = px.launch_app()
register()

借助 OpenInference 自动埋点，只需一行代码即可使用 OpenTelemetry 捕获 Groq API 请求：

`GroqInstrumentor().instrument()`

In [ ]:
from openinference.instrumentation.groq import GroqInstrumentor

GroqInstrumentor().instrument()

## 3. 向 Groq 托管的 LLM 发送请求

以下代码实例化 Groq 客户端，并向 Groq 托管的 __mixtral-8x7b-32768__ LLM 发送请求。OpenInference 自动埋点会捕获请求和响应，并发送至指定端点，本例中为本地 Phoenix 服务。

In [ ]:
from groq import Groq

# Groq 客户端自动读取 API 密钥
client = Groq()

questions = [
    "Explain the importance of low latency LLMs",
    "What is Arize Phoenix and why should I use it?",
    "Is Groq less expensive than hosting on Amazon S3?",
]

# 通过 Groq 客户端发送给 LLM 的请求会追踪到 Phoenix
chat_completions = [
    client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": question,
            }
        ],
        model="mixtral-8x7b-32768",
    )
    for question in questions
]

for chat_completion in chat_completions:
    print("\n------\n" + chat_completion.choices[0].message.content)

# 如果尚未打开 Phoenix 界面，请现在打开

In [ ]:
print("若尚未启动，请在此处启动 Phoenix： ", session.url)